# Notebook 2: Hub Ingestion Walkthrough

This notebook teaches the central hub side of the Phase 1 mock system. It consumes the `payload.msgpack` file created by Notebook 1, sends it through the FastAPI ingestion endpoint, writes the hub SQLite database, and runs the watchdog.

The production scripts behind this notebook are:

- [`central_hub_mock/src/ingestion_api.py`](../central_hub_mock/src/ingestion_api.py)
- [`central_hub_mock/src/init_master_db.py`](../central_hub_mock/src/init_master_db.py)
- [`central_hub_mock/src/watchdog_alert.py`](../central_hub_mock/src/watchdog_alert.py)

## Hub Data Flow

```mermaid
flowchart LR
    A[payload.msgpack
from Notebook 1] --> B[FastAPI POST /ingest_batch]
    B --> C{X-API-Key OK?}
    C -- no --> D[403 rejected]
    C -- yes --> E[MessagePack decode]
    E --> F[Pydantic validation]
    F --> G[SQLite transaction]
    G --> H[ingestion_batches]
    G --> I[hub_buffer_events]
    G --> J[hub_retained_audio_clips]
    G --> K[hub_embedding_segments]
    G --> L[hub_perch_vectors]
    G --> M[health_metrics]
    M --> N[watchdog_alert.py]
```


In [ ]:
from pathlib import Path
import copy
import os
import json
import sqlite3
import sys

import msgpack
import pandas as pd
import yaml
from fastapi.testclient import TestClient

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "central_hub_mock").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

pd.set_option("display.max_colwidth", 120)
print("Repository root:", REPO_ROOT)

In [ ]:
from mock_common.config import load_config
from central_hub_mock.src.init_master_db import init_master_db
from central_hub_mock.src.watchdog_alert import check_watchdog


## 1. Locate The Edge Payload

If this cell fails, run Notebook 1 first. The hub notebook deliberately depends on the edge notebook's output so students can see the handoff between devices.

In [ ]:
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "output"
PAYLOAD_PATH = OUTPUT_ROOT / "transport" / "payload.msgpack"
SUMMARY_PATH = OUTPUT_ROOT / "transport" / "payload_summary.json"

if not PAYLOAD_PATH.exists():
    raise FileNotFoundError("Run notebooks/01_edge_capture_walkthrough.ipynb first; payload.msgpack is missing.")

payload_bytes = PAYLOAD_PATH.read_bytes()
payload_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
print("Payload path:", PAYLOAD_PATH)
print("Payload bytes:", len(payload_bytes))
payload_summary

## 2. Decode MessagePack And Validate Shape

The API receives bytes, decodes MessagePack, and then validates the resulting dictionary with Pydantic models.

In [ ]:
decoded_payload = msgpack.unpackb(payload_bytes, raw=False)
print("Top-level keys:", decoded_payload.keys())
print("Detection count:", len(decoded_payload["detections"]))
print("Telemetry keys:", decoded_payload["telemetry"].keys())
print("First detection segments:", len(decoded_payload["detections"][0]["embedding_segments"]))
print("First embedding bytes:", len(decoded_payload["detections"][0]["embedding_segments"][0]["embedding"]))


## 3. Build A Notebook-Specific Hub Config

The hub database in this notebook lives under `notebooks/output/hub/`. That keeps the teaching database separate from the normal mock database.

In [ ]:
HUB_OUTPUT = OUTPUT_ROOT / "hub"
HUB_OUTPUT.mkdir(parents=True, exist_ok=True)

base_hub_config = load_config(REPO_ROOT / "central_hub_mock" / "config" / "hub_config.example.yaml")
hub_config = copy.deepcopy(base_hub_config)
hub_config.update(
    {
        "master_db_path": str(HUB_OUTPUT / "hub_notebook.sqlite"),
        "api_key": "notebook-dev-key",
        "allowed_device_ids": ["pi_01"],
    }
)
HUB_CONFIG_PATH = HUB_OUTPUT / "hub_config.notebook.yaml"
HUB_CONFIG_PATH.write_text(yaml.safe_dump(hub_config, sort_keys=False), encoding="utf-8")
init_master_db(HUB_CONFIG_PATH, reset=True)

# ingestion_api creates a module-level FastAPI app at import time, so point it at
# the notebook config first. That keeps generated teaching files in notebooks/output/.
os.environ["HUB_CONFIG"] = str(HUB_CONFIG_PATH)
from central_hub_mock.src.ingestion_api import IngestBatchPayload, create_app, load_sqlite_vec

validated_payload = IngestBatchPayload.model_validate(decoded_payload)
print("Validated device:", validated_payload.device_id)
print("Validated detections:", len(validated_payload.detections))
print("Hub config:", HUB_CONFIG_PATH)
print("Hub DB:", hub_config["master_db_path"])


## 4. Send The Payload Through The FastAPI App

For a notebook, `TestClient` is easier than starting a live server. It still exercises the same FastAPI route, API-key check, MessagePack decode, Pydantic validation, and database insert code.

In [ ]:
client = TestClient(create_app(HUB_CONFIG_PATH))

valid_response = client.post(
    "/ingest_batch",
    content=payload_bytes,
    headers={"X-API-Key": hub_config["api_key"]},
)
print("Valid status:", valid_response.status_code)
print(valid_response.json())

bad_key_response = client.post(
    "/ingest_batch",
    content=payload_bytes,
    headers={"X-API-Key": "wrong-key"},
)
print("Bad key status:", bad_key_response.status_code, bad_key_response.json())

malformed_response = client.post(
    "/ingest_batch",
    content=b"not-msgpack",
    headers={"X-API-Key": hub_config["api_key"]},
)
print("Malformed status:", malformed_response.status_code, malformed_response.json())

## 5. Inspect The Hub Database

One accepted batch should create:

- one `ingestion_batches` row,
- one `hub_buffer_events` row per edge buffer,
- three `hub_embedding_segments` rows per buffer,
- three vector rows per buffer,
- one `health_metrics` row for the sender telemetry.

In [ ]:
hub_db_path = Path(hub_config["master_db_path"])
with sqlite3.connect(hub_db_path) as conn:
    meta = dict(conn.execute("SELECT key, value FROM schema_metadata;").fetchall())
    if meta["vector_table"] == "hub_perch_vectors":
        load_sqlite_vec(conn)
    counts = {
        "ingestion_batches": conn.execute("SELECT COUNT(*) FROM ingestion_batches;").fetchone()[0],
        "hub_buffer_events": conn.execute("SELECT COUNT(*) FROM hub_buffer_events;").fetchone()[0],
        "hub_embedding_segments": conn.execute("SELECT COUNT(*) FROM hub_embedding_segments;").fetchone()[0],
        meta["vector_table"]: conn.execute(f"SELECT COUNT(*) FROM {meta['vector_table']};").fetchone()[0],
        "health_metrics": conn.execute("SELECT COUNT(*) FROM health_metrics;").fetchone()[0],
    }
counts

In [ ]:
with sqlite3.connect(hub_db_path) as conn:
    batch_df = pd.read_sql_query("SELECT * FROM ingestion_batches;", conn)
    buffers_df = pd.read_sql_query(
        """
        SELECT hub_buffer_id, device_id, source_buffer_id, retention_reason,
               audio_saved, retained_clip_count, max_nz_bird_common_name,
               ROUND(max_nz_bird_logit, 3) AS max_nz_bird_logit,
               max_perch_label, ROUND(max_perch_logit, 3) AS max_perch_logit
        FROM hub_buffer_events
        ORDER BY source_buffer_id;
        """,
        conn,
    )
    clips_df = pd.read_sql_query(
        """
        SELECT hub_buffer_id, retention_index, filepath, start_offset_s,
               end_offset_s, duration_s, triggered_frame_count
        FROM hub_retained_audio_clips
        ORDER BY hub_buffer_id, retention_index;
        """,
        conn,
    )
    health_df = pd.read_sql_query("SELECT * FROM health_metrics;", conn)

display(batch_df)
display(buffers_df)
display(clips_df)
display(health_df)


## 6. Run The Watchdog

The watchdog is intentionally passive. It looks at the latest telemetry timestamp and decides whether the device is healthy, stale, or missing.

In [ ]:
watchdog_result = check_watchdog(HUB_CONFIG_PATH)
print("Status:", watchdog_result.status)
print("Message:", watchdog_result.message)
print("Age minutes:", watchdog_result.age_minutes)

## 7. Write A Hub Manifest

Notebook 3 will use this manifest to show that hub ingestion happened and to compare edge and hub counts.

In [ ]:
hub_manifest = {
    "created_by": "02_hub_ingestion_walkthrough.ipynb",
    "payload_msgpack": str(PAYLOAD_PATH.relative_to(REPO_ROOT)),
    "hub_config": str(HUB_CONFIG_PATH.relative_to(REPO_ROOT)),
    "hub_db": str(hub_db_path.relative_to(REPO_ROOT)),
    "counts": counts,
    "accepted_buffer_ids": valid_response.json().get("accepted_buffer_ids", []),
    "watchdog": {
        "status": watchdog_result.status,
        "message": watchdog_result.message,
        "age_minutes": watchdog_result.age_minutes,
    },
}
manifest_path = HUB_OUTPUT / "hub_artifact_manifest.json"
manifest_path.write_text(json.dumps(hub_manifest, indent=2) + "\n", encoding="utf-8")
hub_manifest

## What To Inspect Before Notebook 3

```text
notebooks/output/hub/hub_notebook.sqlite
notebooks/output/hub/hub_artifact_manifest.json
notebooks/output/transport/payload.msgpack
```

Notebook 3 will run the whole path again as a system-level rehearsal and compare the key counts.